# Explore here

In [ ]:
import requests       # cargamos la librería que sabe hablar con servidores web (hacer peticiones)
import pandas as pd   # cargamos pandas, que usaremos luego para construir tablas

# Guardamos la dirección a la que vamos a preguntar: temporada 2024, carrera nº1, resultados
url = "https://api.jolpi.ca/ergast/f1/2024/1/results.json"

# Hacemos la petición: "oye servidor, mándame lo que hay en esta URL"
# timeout=30 -> si no contesta en 30 segundos, dejamos de esperar y avisamos de error en vez de quedarnos colgados
response = requests.get(url, timeout=30)

# COMPROBACIÓN DE SEGURIDAD. Si el servidor respondió con un error
# (por ejemplo, error 404 porque la URL está mal, o 500 porque el servidor falló),
# esta línea PARA el programa aquí mismo, con un mensaje claro de qué ha fallado.
# Sin esto, si hubiera un error, el programa seguiría adelante con datos vacíos o rotos,
# y el fallo "de verdad" aparecería 10 líneas más abajo de forma rara y confusa.
response.raise_for_status()

# La respuesta llega como texto con formato JSON. .json() lo convierte en un diccionario
# de Python (como cajones dentro de cajones) para que podamos navegarlo con corchetes ["..."]
data = response.json()

# Vamos abriendo cajones: data -> "MRData" -> "RaceTable" -> "Races"
# "Races" es una LISTA (por eso el [0] al final). Como solo pedimos 1 carrera, solo hay 1 elemento dentro.
carrera = data["MRData"]["RaceTable"]["Races"][0]

# Imprimimos campos sueltos solo para comprobar visualmente que hemos entendido bien la estructura
print("Carrera:", carrera["raceName"])
print("Circuito:", carrera["Circuit"]["circuitName"])
print("Resultado del primer piloto:", carrera["Results"][0])

Carrera: Bahrain Grand Prix
Circuito: Bahrain International Circuit
Resultado del primer piloto: {'number': '1', 'position': '1', 'positionText': '1', 'points': '26', 'Driver': {'driverId': 'max_verstappen', 'permanentNumber': '3', 'code': 'VER', 'url': 'http://en.wikipedia.org/wiki/Max_Verstappen', 'givenName': 'Max', 'familyName': 'Verstappen', 'dateOfBirth': '1997-09-30', 'nationality': 'Dutch'}, 'Constructor': {'constructorId': 'red_bull', 'url': 'https://en.wikipedia.org/wiki/Red_Bull_Racing', 'name': 'Red Bull', 'nationality': 'Austrian'}, 'grid': '1', 'laps': '57', 'status': 'Finished', 'Time': {'millis': '5504742', 'time': '1:31:44.742'}, 'FastestLap': {'rank': '1', 'lap': '39', 'Time': {'time': '1:32.608'}, 'AverageSpeed': {'units': 'kph', 'speed': '210.383'}}}


In [7]:
import requests
import pandas as pd
import time

BASE_URL = "https://api.jolpi.ca/ergast/f1"


def get_json(url, params=None, intentos=8):
    """Pide datos a la API. Si nos frena (429), espera y reintenta solo."""
    espera = 5  # segundos que esperaremos la primera vez que nos frenen

    for intento in range(intentos):
        response = requests.get(url, params=params, timeout=30)

        if response.status_code == 429:
            print(f"Límite alcanzado, esperando {espera}s antes de reintentar...")
            time.sleep(espera)
            espera *= 2
            continue

        response.raise_for_status()
        time.sleep(0.3)
        return response.json()

    raise Exception(f"No se pudo completar la petición tras {intentos} intentos: {url}")

In [8]:
def obtener_resultados(season, ronda):
    """Una fila por piloto con los datos de la carrera."""
    url = f"{BASE_URL}/{season}/{ronda}/results.json"
    data = get_json(url)

    races = data["MRData"]["RaceTable"]["Races"]
    if not races:
        return []

    carrera = races[0]
    filas = []
    for resultado in carrera["Results"]:
        filas.append({
            "season": season,
            "round": ronda,
            "circuitId": carrera["Circuit"]["circuitId"],
            "driverId": resultado["Driver"]["driverId"],
            "driver_nationality": resultado["Driver"]["nationality"],
            "constructorId": resultado["Constructor"]["constructorId"],
            "constructor_nationality": resultado["Constructor"]["nationality"],
            "grid": resultado["grid"],
            "position": resultado["position"],   # target
            "status": resultado["status"],
        })
    return filas


def obtener_clasificacion(season, ronda):
    """Una fila por piloto con los datos de clasificación de esa carrera."""
    url = f"{BASE_URL}/{season}/{ronda}/qualifying.json"
    data = get_json(url)

    races = data["MRData"]["RaceTable"]["Races"]
    if not races:
        return []

    quali = races[0]
    filas = []
    for resultado in quali["QualifyingResults"]:
        filas.append({
            "season": season,
            "round": ronda,
            "driverId": resultado["Driver"]["driverId"],
            "quali_position": resultado["position"],
            "Q1": resultado.get("Q1"),
            "Q2": resultado.get("Q2"),
            "Q3": resultado.get("Q3"),
        })
    return filas


def obtener_vueltas(season, ronda):
    """Una fila por piloto y por vuelta."""
    filas = []
    offset = 0
    limite = 100

    while True:
        url = f"{BASE_URL}/{season}/{ronda}/laps.json"
        data = get_json(url, params={"limit": limite, "offset": offset})

        races = data["MRData"]["RaceTable"]["Races"]
        if not races:
            break

        for vuelta in races[0]["Laps"]:
            for tiempo in vuelta["Timings"]:
                filas.append({
                    "season": season,
                    "round": ronda,
                    "lap": vuelta["number"],
                    "driverId": tiempo["driverId"],
                    "position_en_vuelta": tiempo["position"],
                    "lap_time": tiempo["time"],
                })

        total_registros = int(data["MRData"]["total"])
        offset += limite
        if offset >= total_registros:
            break

    return filas


def numero_de_rondas(season):
    """Cuántas carreras tuvo esa temporada."""
    url = f"{BASE_URL}/{season}.json"
    data = get_json(url)
    return int(data["MRData"]["total"])

In [ ]:
temporadas = [2022, 2023, 2024]

resultados_totales = []
clasificacion_totales = []
vueltas_totales = []

for season in temporadas:
    n_rondas = numero_de_rondas(season)
    print(f"Temporada {season}: {n_rondas} carreras")

    for ronda in range(1, n_rondas + 1):
        resultados_totales += obtener_resultados(season, ronda)
        clasificacion_totales += obtener_clasificacion(season, ronda)
        vueltas_totales += obtener_vueltas(season, ronda)

print("Filas de resultados:", len(resultados_totales))
print("Filas de clasificación:", len(clasificacion_totales))
print("Filas de vueltas:", len(vueltas_totales))

Temporada 2022: 22 carreras
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...
Límite alcanzado, esperando 5s antes de reintentar...


In [ ]:
df_resultados = pd.DataFrame(resultados_totales)
df_clasificacion = pd.DataFrame(clasificacion_totales)
df_vueltas = pd.DataFrame(vueltas_totales)

df_carrera_completa = df_resultados.merge(
    df_clasificacion, on=["season", "round", "driverId"], how="left"
)

df_final = df_vueltas.merge(
    df_carrera_completa, on=["season", "round", "driverId"], how="left"
)

print("Filas finales del dataset:", len(df_final))
df_final.head()

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///f1_dataset.db")
df_final.to_sql("f1_dataset", engine, if_exists="replace", index=False)

# Comprobación final: leemos de vuelta desde la propia base de datos
pd.read_sql("SELECT COUNT(*) AS total_filas FROM f1_dataset", engine)

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("sqlite:///f1_dataset.db")
df = pd.read_sql("SELECT * FROM f1_dataset", engine)

print(len(df))
df.head()